Connect Jupyter to MySQL

In [1]:
import numpy as np
import pandas as pd
import re
from sqlalchemy import create_engine


DB_USER = "root"
DB_PASS="password123"
DB_HOST ="localhost"
DB_PORT="3306"
DB_NAME="Airbnb_Analytics"

engine=create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}: {DB_PORT}/{DB_NAME}")

df=pd.read_sql("Select * from v_cleaned_airbnb_stage ", con=engine)
print(f"Staging recoeds imported : {len(df)}")

Staging recoeds imported : 48895


Advanced Regex Cleaning & Group Median Imputation

In [2]:
def clean_text_field(text):
    if not text or text=='Anonymous Host':
        return 'Anonymous Host'
    cleaned= re.sub(r'[^\w\s]', '', str(text)).strip()
    return cleaned.title() if cleaned else 'Anonymous Host'

df['host_name']=df['host_name'].apply(clean_text_field)
df['neighbourhood']=df['neighbourhood'].apply(clean_text_field)

df['clean_price']=df.groupby(['borough','room_type'])['raw_price'].transform(lambda x:x.fillna(x.median()))

df.drop(columns=['raw_price'],inplace=True)
print("Text standardization and Price Imputation is complete")

Text standardization and Price Imputation is complete


Statistical Outlier Filtering using IQR Method

In [3]:
# 1. Calculate Q1, Q3, and IQR per borough 
Q1 = df.groupby('borough')['clean_price'].transform(lambda x: x.quantile(0.25))
Q3 = df.groupby('borough')['clean_price'].transform(lambda x: x.quantile(0.75))
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 2. Filter outliers
df_sanitized = df[(df['clean_price'] >= lower_bound) & (df['clean_price'] <= upper_bound)].copy()


In [4]:
print("Pipeline executed cleanly! Sanitized shape:", df_sanitized.shape)

Pipeline executed cleanly! Sanitized shape: (46202, 10)


In [5]:
df_sanitized.to_sql(
    name="master_clean_airbnb_listings",
    con=engine,
    if_exists='replace',
    index = False
)
print("Master clean table successfully generated in MySQL.")

Master clean table successfully generated in MySQL.
